# Les Dossiers de l'Inspecteur : jouer

Deux enquêtes dans le Paris des années folles, où chaque personnage est joué par le modèle de langage du projet (Qwen3-4B adapté par LoRA) :

- **Le Dernier Verre du Faisan Doré** (1928), l'affaire sur laquelle l'adaptateur a été entraîné ;
- **Dernière au Théâtre du Miroir** (1929), une intrigue que le modèle n'a jamais vue.

Choisir un GPU (*Exécution > Modifier le type d'exécution > GPU T4*), puis exécuter les trois cellules. Sans GPU, le jeu passe en **mode démo** : les personnages répondent alors avec leur fiche, sans modèle de langage.

In [ ]:
import os, subprocess, sys
from pathlib import Path

DEPOT, BRANCHE = "https://github.com/Neptuniste/agent-enqueteur.git", "enquete-2-et-interface"
DANS_COLAB = "google.colab" in sys.modules
if not Path("src/jeu").exists():                        # notebook ouvert hors du dépôt (Colab)
    if not Path("agent-enqueteur").exists():
        url = DEPOT
        try:                                             # dépôt privé : jeton dans les secrets Colab
            from google.colab import userdata
            url = DEPOT.replace("https://", f"https://{userdata.get('GITHUB_TOKEN')}@")
        except Exception:
            pass
        subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCHE, url, "agent-enqueteur"], check=True)
    os.chdir("agent-enqueteur")
if DANS_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
sys.path.insert(0, os.getcwd())
print("Projet :", os.getcwd())

In [ ]:
import torch
from src.config import ADAPTER_REPO, MODEL_NAME
from src.llm import charger_modele

GPU = torch.cuda.is_available()
ADAPTATEUR = "adapter" if Path("adapter/adapter_config.json").exists() else ADAPTER_REPO   # ou un dépôt Hugging Face
model = tokenizer = None
if GPU:
    model, tokenizer = charger_modele(MODEL_NAME, adaptateur=ADAPTATEUR or None)
    print("Modèle :", MODEL_NAME, "| adaptateur :", ADAPTATEUR or "aucun (modèle de base)")
else:
    print("Pas de GPU : le jeu se lance en mode démo.")

In [ ]:
from src.jeu import lancer_jeu

affiche = MODEL_NAME.split("/")[-1] + (" + LoRA" if ADAPTATEUR else "")
lancer_jeu(model, tokenizer, demo=not GPU, modele_affiche=affiche, share=True)